# EEG Model Colab Training

이 노트북은 MI-Wheelchair-Control 저장소의 `models/` 코드를 Colab GPU에서 실행하기 위한 파일입니다.

실행 전 Colab 메뉴에서 `런타임` -> `런타임 유형 변경` -> `하드웨어 가속기: GPU`를 선택하세요.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def run(cmd):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

print("Python:", sys.version)
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("Torch is not installed yet.")

## 1. Google Drive 연결

아래 셀을 실행하면 Google Drive 권한 승인 창이 나옵니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. GitHub 저장소 가져오기

이미 clone되어 있으면 최신 `main`으로 업데이트합니다.

In [ ]:
REPO_URL = "https://github.com/itsjiyunbro/MI-Wheelchair-Control.git"
REPO_DIR = Path("/content/MI-Wheelchair-Control")

if REPO_DIR.exists():
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"])
else:
    run(["git", "clone", REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print("Current directory:", Path.cwd())

## 3. 전처리 데이터와 저장 경로 설정

아래 후보 경로 중 실제 전처리 폴더가 있는 경로를 사용합니다. 후보에 없으면 `DATA_DIR_CANDIDATES` 첫 줄에 실제 경로를 직접 넣으세요.

In [ ]:
DATA_DIR_CANDIDATES = [
    "/content/drive/MyDrive/MI-Wheelchair/BCI_EEG_Preprocessed_Data_103Subjects",
    "/content/drive/MyDrive/BCI_EEG_Preprocessed_Data_103Subjects",
    "/content/drive/MyDrive/preprocessing_folder/BCI_EEG_Preprocessed_Data_103Subjects",
    "/content/drive/MyDrive/codex/preprocessing_folder/BCI_EEG_Preprocessed_Data_103Subjects",
]
RESULTS_DIR = "/content/drive/MyDrive/MI-Wheelchair-Control/results"
CHECKPOINTS_DIR = "/content/drive/MyDrive/MI-Wheelchair-Control/checkpoints"

DATA_DIR = None
for candidate in DATA_DIR_CANDIDATES:
    candidate_path = Path(candidate)
    print("Checking:", candidate_path)
    if candidate_path.exists():
        DATA_DIR = str(candidate_path)
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find BCI_EEG_Preprocessed_Data_103Subjects in the candidate paths. "
        "Edit DATA_DIR_CANDIDATES in this cell and run it again."
    )

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINTS_DIR).mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("CHECKPOINTS_DIR:", CHECKPOINTS_DIR)

## 4. 데이터 연결 확인

전처리 배열 shape과 label 분포를 먼저 확인합니다.

In [ ]:
run([
    sys.executable,
    "models/scripts/check_data.py",
    "--config", "models/config/default_config.json",
    "--data-dir", DATA_DIR,
    "--results-dir", RESULTS_DIR,
    "--checkpoints-dir", CHECKPOINTS_DIR,
])

## 5. EEGNet 학습

In [ ]:
EEGNET_EPOCHS = 50

run([
    sys.executable,
    "models/scripts/train_eegnet.py",
    "--config", "models/config/default_config.json",
    "--data-dir", DATA_DIR,
    "--results-dir", RESULTS_DIR,
    "--checkpoints-dir", CHECKPOINTS_DIR,
    "--epochs", str(EEGNET_EPOCHS),
])

## 6. ShallowConvNet 학습

In [ ]:
SHALLOW_EPOCHS = 50

run([
    sys.executable,
    "models/scripts/train_shallow_convnet.py",
    "--config", "models/config/default_config.json",
    "--data-dir", DATA_DIR,
    "--results-dir", RESULTS_DIR,
    "--checkpoints-dir", CHECKPOINTS_DIR,
    "--epochs", str(SHALLOW_EPOCHS),
])

## 7. 결과 확인

결과 JSON과 best checkpoint 위치를 확인합니다.

In [ ]:
import json

for name in ["eegnet", "shallow_convnet"]:
    metrics_path = Path(RESULTS_DIR) / f"{name}_metrics.json"
    history_path = Path(RESULTS_DIR) / f"{name}_history.csv"
    checkpoint_path = Path(CHECKPOINTS_DIR) / f"{name}_best.pt"
    print("\nModel:", name)
    print("Metrics:", metrics_path)
    if metrics_path.exists():
        print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    else:
        print("Metrics file was not created.")
    print("History:", history_path, "exists:", history_path.exists())
    print("Checkpoint:", checkpoint_path, "exists:", checkpoint_path.exists())